# <span style="color: #1F1DB5;">SPRINT 18 – Deep Learning y Visión Artificial

# <span style="color: #1F1DB5;">Notebook 1 – Redes Neuronales Artificiales (ANN) y CNN para Imágenes — Versión estudiante

## Cómo usar esta versión estudiante

Este notebook está preparado para que tomes apuntes, completes ejercicios y documentes tus decisiones durante la clase.

**Modo de trabajo recomendado:**

1. Lee primero la consigna de cada bloque o ejercicio.
2. Completa los espacios de respuesta antes de comparar con la explicación del instructor/a.
3. Ejecuta las celdas de setup cuando existan.
4. Escribe interpretación, dudas y decisiones en los espacios de apuntes.
5. Al final, revisa si tus respuestas conectan datos, método e implicación de negocio.

> Las salidas de ejecución fueron limpiadas para que puedas correr el notebook desde cero.


## Notas de clase principales

- Comprender la arquitectura de una red neuronal: capas de entrada, ocultas y salida.
- Entender las funciones de activación (ReLU, sigmoid, softmax) y cuándo usar cada una.
- Comprender forward propagation y backpropagation de manera intuitiva.
- Construir una CNN (Convolutional Neural Network) con Keras para clasificación de imágenes.
- Evaluar modelos de clasificación de imágenes con accuracy y matriz de confusión.

### Mis notas iniciales

- 
- 
- 


## <span style="color: #2826DE;">Objetivos de Aprendizaje

- Comprender la **arquitectura de una red neuronal**: capas de entrada, ocultas y salida.
- Entender las **funciones de activación** (ReLU, sigmoid, softmax) y cuándo usar cada una.
- Comprender **forward propagation y backpropagation** de manera intuitiva.
- Construir una **CNN** (Convolutional Neural Network) con Keras para clasificación de imágenes.
- Evaluar modelos de clasificación de imágenes con accuracy y matriz de confusión.

### <span style="color: #2772F2;">Agenda (2 horas)

| Tema | Duración |
|---|---|
Introducción + Pregunta guía | 10 min |
Arquitectura de una ANN | 20 min |
Funciones de activación | 15 min |
CNN: Convolución y Pooling | 25 min |
Implementación con Keras | 20 min |
Breakout Rooms | 15 min |
Cierre y Kahoot | 5 min |

## <span style="color: #2826DE;">❓ Pregunta Guía

### ¿Cómo puede una computadora "ver" y entender una imagen?

Cuando tú ves una foto de un gato, tu cerebro procesa millones de señales visuales en capas: primero detecta bordes, luego formas, luego texturas, y finalmente reconoce "es un gato". Las redes neuronales convolucionales hacen exactamente lo mismo, capa por capa.

### Mi respuesta inicial

- 


## <span style="color: #2826DE;">Arquitectura de una Red Neuronal Artificial (20 mins)

Una **Red Neuronal Artificial (ANN)** está inspirada en el cerebro humano. Se compone de:

**1. Capa de Entrada (Input Layer):**
- Recibe los datos (features). Un nodo por cada feature.
- Para una imagen de 28×28 píxeles: 784 nodos de entrada.

**2. Capas Ocultas (Hidden Layers):**
- Donde ocurre el "aprendizaje". Cada neurona:
  1. Recibe inputs de la capa anterior.
  2. Los multiplica por **pesos** (weights).
  3. Suma todo + un **sesgo** (bias).
  4. Aplica una **función de activación**.
- Más capas = modelo más profundo = "deep" learning.

**3. Capa de Salida (Output Layer):**
- Produce la predicción final.
- Clasificación binaria: 1 neurona con sigmoid.
- Clasificación multi-clase: N neuronas con softmax.
- Regresión: 1 neurona sin activación.

**Forward Propagation:** Los datos fluyen de entrada → ocultas → salida. Cada capa transforma los datos.

**Backpropagation:** El error de la predicción se propaga "hacia atrás" para ajustar los pesos. Es como decirle a cada neurona "te equivocaste, ajústate".

**Analogía:** Imagina una fábrica con estaciones de trabajo en serie. Cada estación (capa) transforma el producto parcial. Si el producto final sale defectuoso, le dices a cada estación qué ajustar (backprop).

Visualicemos la arquitectura de una red neuronal simple y cómo fluyen los datos a través de ella.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Simulación de forward propagation en una red simple
np.random.seed(42)

# Red: 3 inputs → 4 hidden → 2 output
# Datos de ejemplo (1 muestra con 3 features)
X = np.array([0.5, 0.8, 0.2])

# Pesos aleatorios (en la práctica se aprenden con backprop)
W1 = np.random.randn(3, 4) * 0.5  # Input → Hidden
b1 = np.zeros(4)
W2 = np.random.randn(4, 2) * 0.5  # Hidden → Output
b2 = np.zeros(2)

# Forward propagation paso a paso
print("=== Forward Propagation ===")
print(f"\n1. Input: {X}")

# Capa oculta
z1 = X @ W1 + b1
print(f"\n2. Capa oculta (antes de activación): {z1.round(3)}")
a1 = np.maximum(0, z1)  # ReLU
print(f"   Capa oculta (después de ReLU): {a1.round(3)}")

# Capa de salida
z2 = a1 @ W2 + b2
print(f"\n3. Capa salida (antes de activación): {z2.round(3)}")
# Softmax para clasificación
exp_z2 = np.exp(z2 - np.max(z2))
softmax = exp_z2 / exp_z2.sum()
print(f"   Capa salida (después de softmax): {softmax.round(3)}")
print(f"\n   Predicción: Clase {np.argmax(softmax)} (probabilidad {softmax.max():.1%})")

# Diagrama conceptual
print("""
\n📊 Arquitectura de nuestra red:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Input (3)  →  Hidden (4, ReLU)  →  Output (2, Softmax)
    [x1]          [h1]                  [Clase 0: {:.1%}]
    [x2]    →     [h2]           →     [Clase 1: {:.1%}]
    [x3]          [h3]
                  [h4]
""".format(softmax[0], softmax[1]))

## <span style="color: #2826DE;">Funciones de Activación (15 mins)

Las funciones de activación introducen **no-linealidad** en la red. Sin ellas, una red de muchas capas se reduciría a una simple regresión lineal.

**ReLU (Rectified Linear Unit):** `f(x) = max(0, x)`
- La más usada en capas ocultas.
- Simple y eficiente: si el valor es negativo → 0, si es positivo → pasa tal cual.
- Resuelve el "vanishing gradient problem".

**Sigmoid:** `f(x) = 1 / (1 + e^(-x))`
- Comprime valores entre 0 y 1.
- Ideal para **clasificación binaria** (capa de salida).
- Problema: gradientes muy pequeños para valores extremos.

**Softmax:** `f(x_i) = e^(x_i) / Σ e^(x_j)`
- Convierte un vector en probabilidades que suman 1.
- Ideal para **clasificación multi-clase** (capa de salida).
- Cada neurona de salida = probabilidad de una clase.

**Regla práctica:**
- Capas ocultas → **ReLU** (siempre).
- Salida binaria → **Sigmoid**.
- Salida multi-clase → **Softmax**.

Grafiquemos las tres funciones de activación para entender visualmente cómo transforman los datos.

In [ ]:
# Visualización de funciones de activación
x = np.linspace(-5, 5, 200)

# ReLU
relu = np.maximum(0, x)
# Sigmoid
sigmoid = 1 / (1 + np.exp(-x))
# Softmax (para un par de valores)
softmax_vals = np.exp(x) / (np.exp(x) + np.exp(0))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(x, relu, color='blue', linewidth=2)
axes[0].axhline(0, color='gray', linestyle='--', alpha=0.3)
axes[0].axvline(0, color='gray', linestyle='--', alpha=0.3)
axes[0].set_title('ReLU: max(0, x)', fontsize=13)
axes[0].set_xlabel('x')
axes[0].set_ylabel('f(x)')
axes[0].set_ylim(-1, 5)

axes[1].plot(x, sigmoid, color='red', linewidth=2)
axes[1].axhline(0.5, color='gray', linestyle='--', alpha=0.3)
axes[1].set_title('Sigmoid: 1/(1+e⁻ˣ)', fontsize=13)
axes[1].set_xlabel('x')
axes[1].set_ylabel('f(x)')

axes[2].plot(x, softmax_vals, color='green', linewidth=2)
axes[2].axhline(0.5, color='gray', linestyle='--', alpha=0.3)
axes[2].set_title('Softmax (una clase vs otra)', fontsize=13)
axes[2].set_xlabel('x')
axes[2].set_ylabel('Probabilidad')

plt.tight_layout()
plt.show()

print("Resumen de uso:")
print("  • Capas ocultas → ReLU (por defecto)")
print("  • Salida binaria → Sigmoid (1 neurona)")
print("  • Salida multi-clase → Softmax (N neuronas)")

Preguntas:

- ¿Qué pasaría si usáramos una función lineal (f(x)=x) como activación en todas las capas?

- ¿Por qué ReLU es preferida sobre sigmoid en capas ocultas?

- ¿Cuántas neuronas de salida necesitamos para clasificar dígitos del 0 al 9?

### Mis respuestas de validación

- 
- 


## <span style="color: #2826DE;">CNN: Convolución y Pooling (25 mins)

Las **Redes Neuronales Convolucionales (CNN)** son el estándar para procesamiento de imágenes. ¿Por qué no usar una red densa normal?

**Problema con redes densas para imágenes:**
- Una imagen de 224×224×3 = 150,528 inputs. Demasiados parámetros.
- No capturan **estructura espacial**: píxeles cercanos están relacionados.
- No son **invariantes a la posición**: un gato en la esquina vs en el centro.

**Solución: Capas convolucionales**

**1. Convolución (Conv2D):**
- Un **filtro** (kernel) pequeño (ej. 3×3) se desliza sobre la imagen.
- En cada posición, calcula un producto punto (suma ponderada).
- Diferentes filtros detectan diferentes patrones: bordes, texturas, formas.
- Primeras capas → bordes simples. Capas profundas → patrones complejos.

**2. Pooling (MaxPooling2D):**
- Reduce la dimensión espacial (ej. de 28×28 a 14×14).
- MaxPooling toma el valor máximo en cada ventana.
- Hace el modelo más robusto a pequeños desplazamientos.
- Reduce parámetros y cómputo.

**3. Flatten + Dense:**
- Después de las capas convolucionales, se "aplana" el resultado.
- Se conecta a capas densas para la clasificación final.

**Arquitectura típica de CNN:**
```
Input → [Conv2D → ReLU → MaxPool] × N → Flatten → Dense → Softmax
```

Construyamos una CNN con Keras para clasificar dígitos escritos a mano (MNIST). Veremos cómo definir la arquitectura y entrenar el modelo.

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical

# Cargar datos MNIST (dígitos 0-9, imágenes 28x28)
(X_train, y_train), (X_test, y_test) = mnist.load_data()

# Preprocesamiento
X_train = X_train.reshape(-1, 28, 28, 1).astype('float32') / 255.0
X_test = X_test.reshape(-1, 28, 28, 1).astype('float32') / 255.0
y_train_cat = to_categorical(y_train, 10)
y_test_cat = to_categorical(y_test, 10)

print(f"Train: {X_train.shape} | Labels: {y_train_cat.shape}")
print(f"Test:  {X_test.shape} | Labels: {y_test_cat.shape}")

# Visualizar ejemplos
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i].squeeze(), cmap='gray')
    ax.set_title(f'Dígito: {y_train[i]}')
    ax.axis('off')
plt.suptitle('Ejemplos del dataset MNIST', fontsize=14)
plt.tight_layout()
plt.show()

## <span style="color: #2826DE;">Implementación con Keras (20 mins)

**Keras** es la API de alto nivel de TensorFlow para construir redes neuronales. Su modelo `Sequential` permite apilar capas de forma intuitiva.

**Pasos para construir y entrenar una CNN:**
1. **Definir arquitectura:** `Sequential()` + `.add()` capas.
2. **Compilar:** Elegir optimizador, función de pérdida y métricas.
3. **Entrenar:** `.fit()` con datos de entrenamiento.
4. **Evaluar:** `.evaluate()` con datos de test.

**Hiperparámetros importantes:**
- `filters`: Número de filtros en cada capa conv (32, 64, 128...).
- `kernel_size`: Tamaño del filtro (3×3 es el estándar).
- `epochs`: Cuántas veces ve todo el dataset.
- `batch_size`: Cuántas muestras procesa antes de actualizar pesos.

Ahora definimos, compilamos y entrenamos nuestra CNN. Usaremos un subconjunto de datos para que sea rápido en clase.

In [ ]:
# Usamos subconjunto para rapidez en clase
X_tr = X_train[:10000]
y_tr = y_train_cat[:10000]
X_te = X_test[:2000]
y_te = y_test_cat[:2000]

# Definir la CNN
model = Sequential([
    # Bloque 1: Convolución + Pooling
    Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=(28, 28, 1)),
    MaxPooling2D(pool_size=(2, 2)),

    # Bloque 2: Convolución + Pooling
    Conv2D(64, kernel_size=(3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),

    # Clasificador
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),              # Regularización: apaga 50% de neuronas al azar
    Dense(10, activation='softmax')  # 10 clases (dígitos 0-9)
])

# Compilar
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Resumen de la arquitectura
model.summary()

# Entrenar
print("\nEntrenando...")
history = model.fit(
    X_tr, y_tr,
    validation_split=0.2,
    epochs=5,
    batch_size=64,
    verbose=1
)

# Evaluar
test_loss, test_acc = model.evaluate(X_te, y_te, verbose=0)
print(f"\n✅ Accuracy en test: {test_acc:.2%}")

In [ ]:
# Visualizar predicciones
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

y_pred = model.predict(X_te, verbose=0).argmax(axis=1)
y_true = y_te.argmax(axis=1)

# Matriz de confusión
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(8, 8))
ConfusionMatrixDisplay(cm, display_labels=range(10)).plot(ax=ax, cmap='Blues')
ax.set_title('Matriz de Confusión - CNN MNIST')
plt.tight_layout()
plt.show()

# Ejemplos de predicciones
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    idx = np.random.randint(len(X_te))
    ax.imshow(X_te[idx].squeeze(), cmap='gray')
    pred = y_pred[idx]
    real = y_true[idx]
    color = 'green' if pred == real else 'red'
    ax.set_title(f'Pred: {pred} | Real: {real}', color=color)
    ax.axis('off')
plt.suptitle('Predicciones (verde=correcto, rojo=error)', fontsize=13)
plt.tight_layout()
plt.show()

## <span style="color: #2826DE;">Actividad práctica – Breakout Rooms (15 mins)

### Espacio de trabajo del estudiante

**Respuesta, decisiones o interpretación:**

- 
- 


### <span style="color: #aa0808;">En esta ocasión, quien compartirá resultados será: (Giremos la ruleta)

**Instrucciones:**

1. Modifiquen la CNN anterior para mejorar el accuracy:
   - Agregar una tercera capa convolucional.
   - Cambiar el número de filtros.
   - Agregar más neuronas en la capa densa.
2. Entrenen por 10 epochs en vez de 5. ¿Mejora?
3. ¿Qué pasa si eliminan el Dropout? ¿Se sobreajusta?
4. Registren sus resultados y comparen con otros equipos.

In [ ]:
# TODO estudiante: completa el código de acuerdo con la consigna anterior.
# Contexto: --- TU CÓDIGO AQUÍ ---


## <span style="color: #2826DE;">Tips y buenas prácticas

- Siempre **normaliza** las imágenes (dividir entre 255) antes de alimentar la red.
- Empieza con arquitecturas simples y ve agregando complejidad solo si necesitas más accuracy.
- **Dropout** es tu mejor amigo contra el overfitting en redes neuronales.
- Usa **validation_split** o un validation set para monitorear overfitting durante entrenamiento.
- `model.summary()` te muestra cuántos parámetros tiene tu modelo. Si son demasiados, simplifica.
- Para datasets más complejos, considera **data augmentation** (rotar, escalar, flip).

## <span style="color: #2826DE;">Cierre y Kahoot (5 mins)

Kahoot - Preguntas sugeridas

1️⃣ ¿Qué hace una capa convolucional (Conv2D)?

- Reduce el tamaño de la imagen
- Aplica filtros que detectan patrones locales 
- Conecta todas las neuronas entre sí
- Aplana la imagen a un vector


2️⃣ ¿Cuál es la función de activación recomendada para capas ocultas?

- Sigmoid
- Softmax
- ReLU 
- Lineal


3️⃣ ¿Qué hace MaxPooling2D?

- Aumenta la resolución de la imagen
- Reduce la dimensión espacial tomando el máximo 
- Agrega más filtros
- Normaliza los valores


4️⃣ ¿Qué función de activación usamos en la capa de salida para clasificación multi-clase?

- ReLU
- Sigmoid
- Softmax 
- Tanh


5️⃣ ¿Qué es backpropagation?

- Cuando los datos fluyen de entrada a salida
- El proceso de ajustar pesos propagando el error hacia atrás 
- Cuando eliminamos neuronas con Dropout
- La normalización de las imágenes


6️⃣ ¿Para qué sirve Dropout en una red neuronal?

- Para acelerar el entrenamiento
- Para prevenir overfitting desactivando neuronas al azar 
- Para aumentar el número de parámetros
- Para convertir imágenes a escala de grises